# PR × mutual_info — Case-study OOD from AD Test output

This notebook imports the matching `AD_Test_Regression` Kaggle output as its domain reference and accepted thresholds. It does not clone GitHub, does not import project `.py` files, and does not rebuild Train/Validation.

## Controlled data flow

AD output (Train+Validation selected reference + thresholds) + `cleaned_Casestudy.csv` → unique valid Case SMILES → fresh four-family features → HStack1024 → frozen scaling/selection/model → pIC50, activity, and IND/OOD for k=3...25 at multipliers 0.5 and 2.0.

Case study has no observed pIC50, so regression accuracy metrics are not reported.

In [ ]:
# STEP 0 — Configuration. Blank paths are auto-discovered under /kaggle/input.
RUN_DATASET = 'PR'
RUN_METHOD = 'mutual_info'
EXPECTED_SELECTED_K = 25
EXPECTED_MODEL_CLASS = 'ElasticNet'
EXPECTED_REFERENCE_TEST_R2 = 0.706200850113676
FROZEN_BUNDLE_FILENAME = 'PR_mutual_info_best_model.pkl'

AD_OUTPUT_DIR = ""             # matching AD_Test_Regression output directory
CASE_STUDY_CSV_PATH = ""       # cleaned_Casestudy.csv
SMILES_CHECKPOINT_PATH = ""
SELFIES_CHECKPOINT_PATH = ""
GRAPH_CHECKPOINT_PATH = ""
FINGERPRINT_CHECKPOINT_PATH = ""
FINGERPRINT_TRANSFORMER_PATH = ""

FROZEN_BUNDLE_PATH = ""
FROZEN_MODEL_PATH = ""
FROZEN_SCALER_PATH = ""
FROZEN_SELECTOR_PATH = ""
USE_EMBEDDED_FROZEN_ARTIFACTS = True

INPUT_SEARCH_ROOTS = ["/kaggle/input"]
OUTPUT_ROOT = f"/kaggle/working/OOD_Regression_{RUN_DATASET}_{RUN_METHOD}"
CASE_ID_COLUMN = "ID"
CASE_SMILES_COLUMN = "Smiles"
ACTIVITY_PIC50_THRESHOLD = 6.0
AD_THRESHOLD_MULTIPLIERS = (0.5, 2.0)
AD_NEIGHBOR_K_VALUES = tuple(range(3, 26))
GRAPH_QUERY_CHUNK_SIZE = 2000

AUTO_INSTALL_MISSING_DEPENDENCIES = True
SKFP_INSTALL_SPEC = "git+https://github.com/plenoi/scikit-finger-plenoi.git@master"

assert AD_THRESHOLD_MULTIPLIERS == (0.5, 2.0)
assert AD_NEIGHBOR_K_VALUES == tuple(range(3, 26))
print("OOD variant:", f"{RUN_DATASET} × {RUN_METHOD}")
print("Expected selected features:", EXPECTED_SELECTED_K)
print("Case-study output:", OUTPUT_ROOT)

In [ ]:
# STEP 0B — Install only missing third-party inference dependencies.
import importlib.util
import os
import subprocess
import sys

os.environ.setdefault("WANDB_MODE", "disabled")
os.environ.setdefault("WANDB_SILENT", "true")

required_modules = {
    "transformers": "transformers==4.48.3",
    "sentencepiece": "sentencepiece",
    "selfies": "selfies",
    "rdkit": "rdkit",
    "deepchem": "deepchem",
    "torch_geometric": "torch-geometric",
    "skfp": SKFP_INSTALL_SPEC,
}
missing_specs = [
    package_spec
    for module_name, package_spec in required_modules.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_specs:
    if not AUTO_INSTALL_MISSING_DEPENDENCIES:
        raise ModuleNotFoundError(
            "Missing dependencies: " + ", ".join(missing_specs)
            + ". Install them or enable AUTO_INSTALL_MISSING_DEPENDENCIES."
        )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_specs])

from transformers import AutoModel, AutoTokenizer

def pin_remote_code_revision(auto_class):
    original = auto_class.from_pretrained
    if getattr(original, "_hstack_revision_pinned", False):
        return
    def pinned(pretrained_model_name_or_path, *args, **kwargs):
        revision = kwargs.get("revision")
        if revision:
            kwargs.setdefault("code_revision", revision)
        return original(pretrained_model_name_or_path, *args, **kwargs)
    pinned._hstack_revision_pinned = True
    auto_class.from_pretrained = pinned

pin_remote_code_revision(AutoTokenizer)
pin_remote_code_revision(AutoModel)
print("Third-party dependencies: OK")
print("Inference device contract: CPU")

In [ ]:
# STEP 0C — Materialize the small frozen final artifacts when no Kaggle input contains them.
# These payloads are binary sklearn/joblib artifacts, not executable project source.
import base64
import hashlib
from pathlib import Path

EMBEDDED_FROZEN_FILES_B64 = {'PR_mutual_info_best_model.pkl': '...'}
EMBEDDED_FROZEN_SHA256 = {'PR_mutual_info_best_model.pkl': 'e4066161152efe7d28d4653068dc7d570b6146e206feab47af0ecdf8f2cc3f9e'}
EMBEDDED_FROZEN_ROOT = (
    Path(OUTPUT_ROOT).expanduser().parent
    / "ad_embedded_frozen_inputs"
    / f"{RUN_DATASET}_{RUN_METHOD}"
)

def materialize_embedded_frozen_files():
    EMBEDDED_FROZEN_ROOT.mkdir(parents=True, exist_ok=True)
    resolved = {}
    for filename, encoded in EMBEDDED_FROZEN_FILES_B64.items():
        data = base64.b64decode(encoded.encode("ascii"))
        actual = hashlib.sha256(data).hexdigest()
        expected = EMBEDDED_FROZEN_SHA256[filename]
        if actual != expected:
            raise RuntimeError(f"Embedded frozen artifact checksum failed: {filename}")
        path = EMBEDDED_FROZEN_ROOT / filename
        if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            path.write_bytes(data)
        resolved[filename] = path.resolve()
    return resolved

EMBEDDED_FROZEN_PATHS = (
    materialize_embedded_frozen_files()
    if USE_EMBEDDED_FROZEN_ARTIFACTS
    else {}
)
print("Embedded frozen fallback:", "ready" if EMBEDDED_FROZEN_PATHS else "disabled")
print("Embedded frozen SHA-256:", EMBEDDED_FROZEN_SHA256)

In [ ]:
# STEP 1 — Resolve Case study, deep checkpoints, matching AD output, and frozen final artifacts.
import hashlib
import json
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

OUTPUT_DIR = Path(OUTPUT_ROOT)
ARTIFACT_DIR = OUTPUT_DIR / "intermediate_artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def configured_path(value):
    return Path(value).expanduser().resolve() if str(value).strip() else None

def discover_exact_file(label, configured, filenames, *, required=True):
    explicit = configured_path(configured)
    if explicit is not None:
        if not explicit.is_file():
            raise FileNotFoundError(f"{label} path does not exist: {explicit}")
        return explicit
    matches = []
    for root_value in INPUT_SEARCH_ROOTS:
        root = Path(root_value).expanduser()
        if root.is_dir():
            for filename in filenames:
                matches.extend(path.resolve() for path in root.rglob(filename) if path.is_file())
    matches = sorted(set(matches))
    if len(matches) == 1:
        return matches[0]
    if not matches and not required:
        return None
    if not matches:
        raise FileNotFoundError(
            f"Could not find {label}; set its path in STEP 0. Expected: {list(filenames)}"
        )
    preview = "\n".join(f"  - {path}" for path in matches[:20])
    raise RuntimeError(f"Multiple candidates for {label}; set the exact path in STEP 0:\n{preview}")

def discover_ad_output_dir(configured):
    explicit = configured_path(configured)
    if explicit is not None:
        metadata_paths = [explicit / "run_metadata.json"] if explicit.is_dir() else []
    else:
        metadata_paths = []
        for root_value in INPUT_SEARCH_ROOTS:
            root = Path(root_value).expanduser()
            if root.is_dir():
                metadata_paths.extend(root.rglob("run_metadata.json"))
    matches = []
    for metadata_path in metadata_paths:
        try:
            payload = json.loads(metadata_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if (
            payload.get("dataset") == RUN_DATASET
            and payload.get("method") == RUN_METHOD
            and payload.get("ad_reference") == "Train + Validation selected features"
        ):
            matches.append(metadata_path.parent.resolve())
    matches = sorted(set(matches))
    if len(matches) != 1:
        preview = "\n".join(f"  - {path}" for path in matches[:20]) or "  (none)"
        raise RuntimeError(
            f"Expected exactly one matching AD output for {RUN_DATASET}/{RUN_METHOD}; "
            f"found {len(matches)}. Set AD_OUTPUT_DIR in STEP 0.\n{preview}"
        )
    return matches[0]

AD_INPUT_DIR = discover_ad_output_dir(AD_OUTPUT_DIR)
INPUT_PATHS = {
    "case_study_csv": discover_exact_file(
        "Case-study CSV", CASE_STUDY_CSV_PATH, ["cleaned_Casestudy.csv"]
    ),
    "smiles_checkpoint": discover_exact_file(
        "SMILES checkpoint", SMILES_CHECKPOINT_PATH, [f"model_smiles_{RUN_DATASET}.pt"]
    ),
    "selfies_checkpoint": discover_exact_file(
        "SELFIES checkpoint", SELFIES_CHECKPOINT_PATH, [f"model_selfies_{RUN_DATASET}.pt"]
    ),
    "graph_checkpoint": discover_exact_file(
        "Graph checkpoint", GRAPH_CHECKPOINT_PATH, [f"model_graph_{RUN_DATASET}.pt"]
    ),
    "fingerprint_checkpoint": discover_exact_file(
        "Fingerprint checkpoint", FINGERPRINT_CHECKPOINT_PATH,
        [f"model_fingerprint_{RUN_DATASET}.pt"],
    ),
    "fingerprint_transformer": discover_exact_file(
        "Fingerprint transformer", FINGERPRINT_TRANSFORMER_PATH,
        [f"ecfp_transformer_{RUN_DATASET}.pkl"],
    ),
}

separate_frozen_paths = all(
    str(value).strip()
    for value in (FROZEN_MODEL_PATH, FROZEN_SCALER_PATH, FROZEN_SELECTOR_PATH)
)
if separate_frozen_paths:
    INPUT_PATHS["frozen_model"] = discover_exact_file(
        "frozen model", FROZEN_MODEL_PATH, [Path(FROZEN_MODEL_PATH).name]
    )
    INPUT_PATHS["frozen_scaler"] = discover_exact_file(
        "frozen scaler", FROZEN_SCALER_PATH, [Path(FROZEN_SCALER_PATH).name]
    )
    INPUT_PATHS["frozen_selector"] = discover_exact_file(
        "frozen selector", FROZEN_SELECTOR_PATH, [Path(FROZEN_SELECTOR_PATH).name]
    )
    FROZEN_INPUT_SOURCE = "three explicit paths"
else:
    if any(str(value).strip() for value in (FROZEN_MODEL_PATH, FROZEN_SCALER_PATH, FROZEN_SELECTOR_PATH)):
        raise ValueError("Provide all three frozen component paths, or leave all three blank")
    discovered_bundle = discover_exact_file(
        "frozen bundle", FROZEN_BUNDLE_PATH, [FROZEN_BUNDLE_FILENAME], required=False
    )
    if discovered_bundle is not None:
        INPUT_PATHS["frozen_bundle"] = discovered_bundle
        FROZEN_INPUT_SOURCE = "external Kaggle input"
    elif USE_EMBEDDED_FROZEN_ARTIFACTS:
        INPUT_PATHS["frozen_bundle"] = EMBEDDED_FROZEN_PATHS[FROZEN_BUNDLE_FILENAME]
        FROZEN_INPUT_SOURCE = "embedded checksum-verified fallback"
    else:
        raise FileNotFoundError(
            f"Frozen bundle {FROZEN_BUNDLE_FILENAME!r} was not found and fallback is disabled"
        )

print("Matching AD output:", AD_INPUT_DIR)
print("Frozen final-artifact source:", FROZEN_INPUT_SOURCE)
display(pd.DataFrame([{"Input": key, "Resolved path": str(value)} for key, value in INPUT_PATHS.items()]))

In [ ]:
# STEP 2 — Load the frozen model, MinMaxScaler, and feature selector.
def _corr_abs(A, b, method):
    if method == "spearman":
        from scipy.stats import rankdata
        A = np.apply_along_axis(rankdata, 0, A)
        b = rankdata(b)
    centered_y = b - b.mean()
    centered_X = A - A.mean(0)
    numerator = (centered_X * centered_y[:, None]).sum(0)
    denominator = np.sqrt((centered_X**2).sum(0)) * np.sqrt((centered_y**2).sum())
    return np.abs(numerator / (denominator + 1e-12))

sys.modules["__main__"]._corr_abs = _corr_abs

def load_frozen_objects():
    if separate_frozen_paths:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model = joblib.load(INPUT_PATHS["frozen_model"])
            scaler = joblib.load(INPUT_PATHS["frozen_scaler"])
            selector = joblib.load(INPUT_PATHS["frozen_selector"])
        metadata = {
            "dataset": RUN_DATASET, "FS_Method": RUN_METHOD,
            "FS_k": EXPECTED_SELECTED_K, "model_name": EXPECTED_MODEL_CLASS,
            "TSR2": EXPECTED_REFERENCE_TEST_R2,
        }
        return model, scaler, selector, metadata

    bundle_path = INPUT_PATHS["frozen_bundle"]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        if bundle_path.suffix.lower() == ".json":
            descriptor = json.loads(bundle_path.read_text(encoding="utf-8"))
            if descriptor.get("format") != "frozen_component_bundle_v1":
                raise AssertionError(f"Unsupported frozen descriptor: {bundle_path}")
            components = descriptor["components"]
            model = joblib.load(bundle_path.parent / components["model"])
            scaler = joblib.load(bundle_path.parent / components["normalizer"])
            selector = joblib.load(bundle_path.parent / components["selector"])
            metadata = dict(descriptor["metadata"])
        else:
            payload = joblib.load(bundle_path)
            model = payload["model"]
            scaler = payload["normalizer"]
            selector = payload["selector"]
            metadata = {key: value for key, value in payload.items() if key not in {"model", "normalizer", "selector"}}
    return model, scaler, selector, metadata

frozen_model, frozen_scaler, frozen_selector, frozen_metadata = load_frozen_objects()
selected_indices = np.asarray(frozen_selector.get_support(indices=True), dtype=int)

checks = {
    "model class": frozen_model.__class__.__name__ == EXPECTED_MODEL_CLASS,
    "selected feature count": len(selected_indices) == EXPECTED_SELECTED_K,
    "scaler input dimension": int(frozen_scaler.n_features_in_) == 1024,
    "selector input dimension": int(frozen_selector.n_features_in_) == 1024,
    "model input dimension": int(frozen_model.n_features_in_) == EXPECTED_SELECTED_K,
}
failed = [name for name, passed in checks.items() if not passed]
if failed:
    raise AssertionError(f"Frozen artifact contract failed: {failed}")

display(pd.DataFrame([{"Check": key, "Passed": value} for key, value in checks.items()]))
print("Model:", frozen_model)
print("Scaler:", frozen_scaler.__class__.__name__)
print("Selector:", frozen_selector.__class__.__name__)
print("Selected HStack1024 indices:", selected_indices.tolist())

In [ ]:
# STEP 3 — Load and validate the exact domain reference and thresholds from AD Test output.
ad_metadata_path = AD_INPUT_DIR / "run_metadata.json"
ad_manifest_path = AD_INPUT_DIR / "artifact_manifest.csv"
ad_artifact_dir = AD_INPUT_DIR / "intermediate_artifacts"
ad_raw_splits_path = ad_artifact_dir / "01_raw_splits.npz"
ad_selected_path = ad_artifact_dir / f"05_selected_features_k{EXPECTED_SELECTED_K}.npz"
ad_reference_path = ad_artifact_dir / "07_ad_reference_train_plus_validation.npy"

required_ad_files = [
    ad_metadata_path, ad_manifest_path, ad_raw_splits_path,
    ad_selected_path, ad_reference_path,
]
missing_ad_files = [path for path in required_ad_files if not path.is_file()]
if missing_ad_files:
    raise FileNotFoundError(f"AD output is incomplete: {missing_ad_files}")

ad_metadata = json.loads(ad_metadata_path.read_text(encoding="utf-8"))
if ad_metadata.get("dataset") != RUN_DATASET or ad_metadata.get("method") != RUN_METHOD:
    raise AssertionError("AD output dataset/method does not match this OOD notebook")
if int(ad_metadata.get("selected_feature_count")) != EXPECTED_SELECTED_K:
    raise AssertionError("AD output selected-feature count changed")
if tuple(ad_metadata.get("ad_k_values", [])) != AD_NEIGHBOR_K_VALUES:
    raise AssertionError("AD output k values changed")
if tuple(float(value) for value in ad_metadata.get("threshold_multipliers", [])) != AD_THRESHOLD_MULTIPLIERS:
    raise AssertionError("AD output threshold multipliers changed")

raw_splits = np.load(ad_raw_splits_path, allow_pickle=False)
ad_selected = np.load(ad_selected_path, allow_pickle=False)
X_reference = np.load(ad_reference_path, allow_pickle=False).astype(np.float32)
ad_selected_indices = np.asarray(ad_selected["selected_indices"], dtype=int)
y_reference = np.concatenate([raw_splits["train_y"], raw_splits["val_y"]]).astype(float)
expected_reference = np.vstack([ad_selected["train"], ad_selected["val"]]).astype(np.float32)

if not np.array_equal(ad_selected_indices, selected_indices):
    raise AssertionError("Frozen selector indices differ from AD Test output")
if X_reference.shape != expected_reference.shape or not np.array_equal(X_reference, expected_reference):
    raise AssertionError("AD reference differs from saved Train+Validation selected features")
if len(y_reference) != len(X_reference):
    raise AssertionError("AD reference rows and Train+Validation targets do not align")

ad_summaries = {}
for multiplier in AD_THRESHOLD_MULTIPLIERS:
    token = "0p5" if multiplier == 0.5 else "2p0"
    path = AD_INPUT_DIR / (
        f"AD_Test_Regression_{RUN_DATASET}_{RUN_METHOD}_threshold_{token}.csv"
    )
    if not path.is_file():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path)
    expected_labels = ["No AD"] + [f"k={k}" for k in AD_NEIGHBOR_K_VALUES]
    if frame["K"].tolist() != expected_labels:
        raise AssertionError(f"Unexpected AD summary rows: {path}")
    ad_summaries[multiplier] = frame

print("AD reference shape:", X_reference.shape)
print("Reference target shape:", y_reference.shape)
print("Selected indices verified:", ad_selected_indices.tolist())
display(pd.DataFrame({
    "Multiplier": list(ad_summaries),
    "Summary rows": [len(frame) for frame in ad_summaries.values()],
}))

In [ ]:
# STEP 4 — Validate every Case-study row, preserve duplicates, and isolate unique valid SMILES.
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")
case_raw = pd.read_csv(INPUT_PATHS["case_study_csv"])
required_case_columns = {CASE_ID_COLUMN, CASE_SMILES_COLUMN}
missing_case_columns = required_case_columns.difference(case_raw.columns)
if missing_case_columns:
    raise ValueError(f"Case-study CSV is missing columns: {sorted(missing_case_columns)}")

case_frame = case_raw.copy()
case_frame.insert(0, "Case_Row", np.arange(len(case_frame), dtype=int))
case_frame["Original_ID"] = case_frame[CASE_ID_COLUMN]
case_frame["Original_SMILES"] = case_frame[CASE_SMILES_COLUMN]
case_frame["Normalized_SMILES"] = case_frame[CASE_SMILES_COLUMN].fillna("").astype(str).str.strip()

def validate_case_smiles(value):
    if not value:
        return "MISSING_SMILES"
    return "VALID" if Chem.MolFromSmiles(value) is not None else "INVALID_SMILES"

case_frame["Input_Status"] = case_frame["Normalized_SMILES"].map(validate_case_smiles)
valid_mask = case_frame["Input_Status"].eq("VALID").to_numpy()
if not valid_mask.any():
    raise ValueError("Case study has no valid SMILES")

valid_smiles = case_frame.loc[valid_mask, "Normalized_SMILES"].to_numpy().astype(str)
unique_codes, unique_smiles = pd.factorize(valid_smiles, sort=False)
unique_smiles = np.asarray(unique_smiles).astype(str)
case_frame["Duplicate_Valid_SMILES"] = False
case_frame.loc[valid_mask, "Duplicate_Valid_SMILES"] = pd.Series(valid_smiles).duplicated(keep=False).to_numpy()

case_audit_path = ARTIFACT_DIR / "01_case_study_input_audit.csv"
unique_path = ARTIFACT_DIR / "02_unique_valid_case_smiles.csv"
case_frame.to_csv(case_audit_path, index=False)
pd.DataFrame({"Unique_Query_Row": np.arange(len(unique_smiles)), "Smiles": unique_smiles}).to_csv(
    unique_path, index=False
)

print("Case-study rows:", len(case_frame))
print("Valid rows:", int(valid_mask.sum()))
print("Invalid/missing rows:", int((~valid_mask).sum()))
print("Unique valid SMILES to featurize:", len(unique_smiles))
print("Duplicate valid rows:", int(case_frame["Duplicate_Valid_SMILES"].sum()))
print("Saved:", case_audit_path)
display(case_frame.head())

## Fresh Case-study feature extraction

The complete frozen extraction implementation is embedded below for audit. Only unique valid Case-study SMILES are computed; results are mapped back to all rows.

In [ ]:
# STEP 5 — Complete frozen feature-extractor definitions for Case study.
# This source is embedded in the notebook for direct inspection; it is not imported from GitHub.
import warnings
import joblib


from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Dict, Optional, Sequence

import numpy as np

FAMILY_DIM = 256

def normalize_dataset(dataset):
    value = str(dataset).upper()
    return "ER" if value == "ERALPHA" else value

def checkpoint_path(dataset, family, repo_root=None):
    return INPUT_PATHS[f"{family}_checkpoint"]

def fingerprint_transformer_path(dataset, repo_root=None):
    return INPUT_PATHS["fingerprint_transformer"]

def load_joblib(path):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return joblib.load(path)

@dataclass(frozen=True)
class FeatureSplit:
    smiles: np.ndarray
    X: np.ndarray
    y: np.ndarray


MOLFORMER_REPO = "ibm/MoLFormer-XL-both-10pct"
MOLFORMER_REVISION = "7b12d946c181a37f6012b9dc3b002275de070314"
BIOT5_REPO = "QizhiPei/biot5-plus-base"
BIOT5_REVISION = "622b6a04bf0fe266c67dd0927374b00bf21eb7de"
MAX_LENGTH = 128
BATCH_SIZE = 32
INFERENCE_DEVICE = "cpu"


def _seed_cpu() -> None:
    import random
    import torch

    random.seed(0)
    np.random.seed(0)
    torch.manual_seed(0)
    torch.use_deterministic_algorithms(True, warn_only=True)


def _device():
    import torch

    # The research-replay contract is CPU-only. Never select CUDA implicitly:
    # different device kernels are outside the locked inference environment.
    return torch.device(INFERENCE_DEVICE)


def _require_256(name: str, features: np.ndarray, n_rows: int) -> np.ndarray:
    features = np.asarray(features, dtype=np.float32)
    if features.shape != (n_rows, FAMILY_DIM):
        raise AssertionError(
            f"{name}: expected {(n_rows, FAMILY_DIM)}, got {features.shape}"
        )
    if not np.isfinite(features).all():
        raise AssertionError(f"{name}: non-finite feature value")
    return features


def _transformer_dataset(sequences: Sequence[str], tokenizer):
    import torch
    from torch.utils.data import Dataset

    class SequenceDataset(Dataset):
        def __len__(self):
            return len(sequences)

        def __getitem__(self, index):
            encoded = tokenizer(
                str(sequences[index]),
                padding="max_length",
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )
            return {key: value.squeeze(0) for key, value in encoded.items()}

    return SequenceDataset()


def _molformer_regressor(state_dict, dataset: str):
    import torch.nn as nn

    first_dim = int(state_dict["regressor.0.weight"].shape[0])
    dropout = 0.25 if normalize_dataset(dataset) == "ER" else 0.2
    return nn.Sequential(
        nn.Linear(768, first_dim),
        nn.LayerNorm(first_dim),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(first_dim, 256),
        nn.LayerNorm(256),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(256, 1),
    )


def extract_smiles(
    dataset: str, smiles: Sequence[str], repo_root: Optional[Path] = None
) -> np.ndarray:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader
    from transformers import AutoConfig, AutoModel, AutoTokenizer

    _seed_cpu()
    device = _device()
    state = torch.load(
        checkpoint_path(dataset, "smiles", repo_root),
        map_location="cpu",
        weights_only=True,
    )
    hub_args = {
        "trust_remote_code": True,
        "revision": MOLFORMER_REVISION,
        # Some repository auto_map entries explicitly name the source repo.
        # Without code_revision, Transformers can load remote Python from the
        # current HEAD while loading weights/config.json from revision, creating
        # two distinct MolformerConfig classes and a registration failure.
        "code_revision": MOLFORMER_REVISION,
    }
    config = AutoConfig.from_pretrained(
        MOLFORMER_REPO, deterministic_eval=True, **hub_args
    )
    tokenizer = AutoTokenizer.from_pretrained(MOLFORMER_REPO, **hub_args)
    base = AutoModel.from_pretrained(MOLFORMER_REPO, config=config, **hub_args)

    class FrozenMolFormer(nn.Module):
        def __init__(self):
            super().__init__()
            self.transformer = base
            self.regressor = _molformer_regressor(state, dataset)

    model = FrozenMolFormer()
    model.load_state_dict(state)
    model.to(device).eval()
    flags = [module.deterministic for module in model.modules() if hasattr(module, "deterministic")]
    if not flags or not all(flags):
        raise AssertionError("MoLFormer deterministic_eval is not active")
    loader = DataLoader(_transformer_dataset(smiles, tokenizer), batch_size=BATCH_SIZE)
    chunks = []
    with torch.no_grad():
        for batch in loader:
            output = model.transformer(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            )
            # Historical protocol: unmasked mean across all 128 token positions.
            pooled = output.last_hidden_state.mean(dim=1)
            chunks.append(model.regressor[:8](pooled).cpu().numpy())
    return _require_256("smiles", np.vstack(chunks), len(smiles))


def extract_selfies(
    dataset: str, smiles: Sequence[str], repo_root: Optional[Path] = None
) -> np.ndarray:
    import selfies as sf
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader
    from transformers import AutoConfig, AutoModel, AutoTokenizer

    _seed_cpu()
    device = _device()
    sequences = []
    for index, value in enumerate(smiles):
        try:
            sequences.append(sf.encoder(str(value)))
        except Exception as exc:
            raise ValueError(f"SELFIES conversion failed at row {index}: {value!r}") from exc
    state = torch.load(
        checkpoint_path(dataset, "selfies", repo_root),
        map_location="cpu",
        weights_only=True,
    )
    hub_args = {
        "trust_remote_code": True,
        "revision": BIOT5_REVISION,
        "code_revision": BIOT5_REVISION,
    }
    config = AutoConfig.from_pretrained(BIOT5_REPO, **hub_args)
    tokenizer = AutoTokenizer.from_pretrained(BIOT5_REPO, **hub_args)
    base = AutoModel.from_pretrained(BIOT5_REPO, config=config, **hub_args)

    class FrozenBioT5(nn.Module):
        def __init__(self):
            super().__init__()
            self.transformer = base
            hidden = int(base.config.hidden_size)
            self.regressor = nn.Sequential(
                nn.Linear(hidden, 512),
                nn.LayerNorm(512),
                nn.GELU(),
                nn.Linear(512, 256),
                nn.LayerNorm(256),
                nn.GELU(),
                nn.Linear(256, 1),
            )

    model = FrozenBioT5()
    model.load_state_dict(state)
    model.to(device).eval()
    loader = DataLoader(_transformer_dataset(sequences, tokenizer), batch_size=BATCH_SIZE)
    chunks = []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            decoder_ids = torch.zeros(
                (ids.size(0), 1), dtype=torch.long, device=device
            )
            output = model.transformer(
                input_ids=ids,
                attention_mask=mask,
                decoder_input_ids=decoder_ids,
            )
            pooled = output.last_hidden_state.mean(dim=1)
            chunks.append(model.regressor[:6](pooled).cpu().numpy())
    return _require_256("selfies", np.vstack(chunks), len(smiles))


def extract_fingerprint(
    dataset: str, smiles: Sequence[str], repo_root: Optional[Path] = None
) -> np.ndarray:
    import torch
    import torch.nn as nn

    _seed_cpu()
    device = _device()
    transformer = load_joblib(fingerprint_transformer_path(dataset, repo_root))
    ecfp = np.asarray(transformer.transform(list(smiles)), dtype=np.float32)
    if ecfp.shape != (len(smiles), 2048):
        raise AssertionError(f"fingerprint: expected {(len(smiles), 2048)}, got {ecfp.shape}")

    class FrozenECFPRegressor(nn.Module):
        def __init__(self):
            super().__init__()
            self.regressor = nn.Sequential(
                nn.Linear(2048, 1024),
                nn.LayerNorm(1024),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(1024, 256),
                nn.LayerNorm(256),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(256, 1),
            )

    model = FrozenECFPRegressor()
    model.load_state_dict(
        torch.load(
            checkpoint_path(dataset, "fingerprint", repo_root),
            map_location="cpu",
            weights_only=True,
        )
    )
    model.to(device).eval()
    with torch.no_grad():
        features = model.regressor[:8](torch.from_numpy(ecfp).to(device)).cpu().numpy()
    return _require_256("fingerprint", features, len(smiles))


def extract_graph(
    dataset: str, smiles: Sequence[str], repo_root: Optional[Path] = None
) -> np.ndarray:
    import deepchem as dc
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from rdkit import Chem
    from torch_geometric.data import Data
    from torch_geometric.loader import DataLoader
    from torch_geometric.nn import ChebConv, global_max_pool, global_mean_pool

    _seed_cpu()
    device = _device()
    featurizer = dc.feat.MolGraphConvFeaturizer(use_edges=True)
    graphs = []
    for index, value in enumerate(smiles):
        smi = str(value)
        try:
            if Chem.MolFromSmiles(smi) is None:
                raise ValueError("invalid RDKit SMILES")
            graph = featurizer.featurize([smi])[0]
            if graph.node_features is None or graph.num_nodes == 0:
                raise ValueError("empty graph")
            graphs.append(
                Data(
                    x=torch.tensor(graph.node_features, dtype=torch.float32),
                    edge_index=torch.tensor(graph.edge_index, dtype=torch.long),
                    edge_attr=torch.tensor(graph.edge_features, dtype=torch.float32),
                )
            )
        except Exception as exc:
            # The original notebooks silently skipped failed graphs.  That is not
            # safe for an aligned four-family pipeline, so fail without reordering.
            raise ValueError(f"Graph conversion failed at row {index}: {smi!r}") from exc

    class FrozenChebRegressorV3(nn.Module):
        def __init__(self):
            super().__init__()
            self.input_proj = nn.Linear(30, 256)
            self.convs = nn.ModuleList([ChebConv(256, 256, 3) for _ in range(3)])
            self.lns = nn.ModuleList([nn.LayerNorm(256) for _ in range(3)])
            self.fc = nn.Sequential(
                nn.Linear(512, 256), nn.LayerNorm(256), nn.GELU(), nn.Linear(256, 1)
            )

    model = FrozenChebRegressorV3()
    model.load_state_dict(
        torch.load(
            checkpoint_path(dataset, "graph", repo_root),
            map_location="cpu",
            weights_only=True,
        )
    )
    model.to(device).eval()
    chunks = []
    with torch.no_grad():
        for batch in DataLoader(graphs, batch_size=BATCH_SIZE, shuffle=False):
            batch = batch.to(device)
            x = F.relu(model.input_proj(batch.x))
            for convolution, layer_norm in zip(model.convs, model.lns):
                x = x + F.relu(layer_norm(convolution(x, batch.edge_index)))
            mean = global_mean_pool(x, batch.batch)
            maximum = global_max_pool(x, batch.batch)
            chunks.append(
                model.fc[:3](torch.cat([mean, maximum], dim=-1)).cpu().numpy()
            )
    return _require_256("graph", np.vstack(chunks), len(smiles))


EXTRACTORS: Dict[str, Callable[..., np.ndarray]] = {
    "smiles": extract_smiles,
    "selfies": extract_selfies,
    "graph": extract_graph,
    "fingerprint": extract_fingerprint,
}


def extract_all(
    dataset: str,
    smiles: Sequence[str],
    targets: Sequence[float],
    repo_root: Optional[Path] = None,
) -> Dict[str, FeatureSplit]:
    ds = normalize_dataset(dataset)
    smi = np.asarray(smiles).astype(str)
    y = np.asarray(targets)
    return {
        family: FeatureSplit(smi, EXTRACTORS[family](ds, smi, repo_root), y)
        for family in ("smiles", "selfies", "graph", "fingerprint")
    }

In [ ]:
# STEP 6 — Prepare Case-study feature containers and reloadable family outputs.
query_family_features = {}

def save_query_family(family, matrix):
    matrix = np.asarray(matrix, dtype=np.float32)
    if matrix.shape != (len(unique_smiles), 256):
        raise AssertionError(f"{family}: unexpected query feature shape {matrix.shape}")
    query_family_features[family] = matrix
    output_path = ARTIFACT_DIR / f"03_case_{family}_features_256.npy"
    np.save(output_path, matrix)
    print(f"{family}: shape={matrix.shape}, first five={matrix[0, :5]}")
    print("Saved:", output_path)

In [ ]:
# STEP 7 — Extract fresh SMILES Case-study features and save them.
smiles_query = extract_smiles(RUN_DATASET, unique_smiles)
save_query_family('smiles', smiles_query)

In [ ]:
# STEP 7 — Extract fresh SELFIES Case-study features and save them.
selfies_query = extract_selfies(RUN_DATASET, unique_smiles)
save_query_family('selfies', selfies_query)

In [ ]:
# STEP 7 — Extract fresh GRAPH Case-study features and save them.
graph_parts = []
for left in range(0, len(unique_smiles), GRAPH_QUERY_CHUNK_SIZE):
    right = min(left + GRAPH_QUERY_CHUNK_SIZE, len(unique_smiles))
    print(f"Graph chunk: rows {left}:{right} of {len(unique_smiles)}")
    graph_parts.append(extract_graph(RUN_DATASET, unique_smiles[left:right]))
graph_query = np.vstack(graph_parts).astype(np.float32)
save_query_family('graph', graph_query)

In [ ]:
# STEP 7 — Extract fresh FINGERPRINT Case-study features and save them.
fingerprint_query = extract_fingerprint(RUN_DATASET, unique_smiles)
save_query_family('fingerprint', fingerprint_query)

In [ ]:
# STEP 8 — HStack1024 → frozen MinMaxScaler → frozen selector for Case study.
FAMILY_ORDER = ("smiles", "selfies", "graph", "fingerprint")
X_query_hstack = np.hstack([query_family_features[name] for name in FAMILY_ORDER]).astype(np.float32)
if X_query_hstack.shape != (len(unique_smiles), 1024):
    raise AssertionError(f"Unexpected Case-study HStack shape: {X_query_hstack.shape}")

X_query_scaled = (
    X_query_hstack * np.asarray(frozen_scaler.scale_) + np.asarray(frozen_scaler.min_)
).astype(np.float32)
X_query_selected = X_query_scaled[:, selected_indices].astype(np.float32)
if X_query_selected.shape[1] != X_reference.shape[1]:
    raise AssertionError("Case-study and AD-reference feature dimensions differ")

hstack_path = ARTIFACT_DIR / "04_case_hstack1024.npy"
scaled_path = ARTIFACT_DIR / "05_case_minmax_scaled_hstack1024.npy"
selected_path = ARTIFACT_DIR / f"06_case_selected_features_k{EXPECTED_SELECTED_K}.npy"
np.save(hstack_path, X_query_hstack)
np.save(scaled_path, X_query_scaled)
np.save(selected_path, X_query_selected)

print("Case HStack1024:", X_query_hstack.shape)
print("Case scaled HStack1024:", X_query_scaled.shape)
print("Case selected features:", X_query_selected.shape)
print("AD reference:", X_reference.shape)
print("Saved:", selected_path)

In [ ]:
# STEP 9 — Frozen pIC50 prediction and Train+Validation-only LDA activity reporting.
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

def predict_frozen_linear(X, model):
    features = np.asarray(X, dtype=np.float64)
    coefficients = np.asarray(model.coef_, dtype=np.float64).reshape(1, -1)
    intercept = np.asarray(model.intercept_, dtype=np.float64)
    return (np.sum(features * coefficients, axis=1) + intercept).reshape(-1)

unique_predicted_pic50 = predict_frozen_linear(X_query_selected, frozen_model)
activity_y = np.where(y_reference >= ACTIVITY_PIC50_THRESHOLD, 0, 1)
if set(np.unique(activity_y)) != {0, 1}:
    raise ValueError("Train+Validation activity labels do not contain both classes")
activity_classifier = LinearDiscriminantAnalysis(tol=0.00001)
activity_classifier.fit(X_reference, activity_y)
unique_activity_class = activity_classifier.predict(X_query_selected)
positive_column = int(np.flatnonzero(activity_classifier.classes_ == 0)[0])
unique_probability = activity_classifier.predict_proba(X_query_selected)[:, positive_column]
unique_activity_label = np.where(unique_activity_class == 0, "Positive", "Negative")

joblib.dump(activity_classifier, OUTPUT_DIR / "activity_classifier_lda.joblib")
unique_predictions = pd.DataFrame({
    "Unique_Query_Row": np.arange(len(unique_smiles), dtype=int),
    "Smiles": unique_smiles,
    "Predicted_pIC50": unique_predicted_pic50,
    "Predicted": unique_activity_label,
    "Probability": unique_probability,
})
unique_prediction_path = ARTIFACT_DIR / "07_unique_case_predictions.csv"
unique_predictions.to_csv(unique_prediction_path, index=False)

full_predictions = case_frame.copy()
full_predictions["Predicted_pIC50"] = np.nan
full_predictions["Predicted"] = "INVALID"
full_predictions["Probability"] = np.nan
valid_positions = np.flatnonzero(valid_mask)
full_predictions.loc[valid_positions, "Predicted_pIC50"] = unique_predicted_pic50[unique_codes]
full_predictions.loc[valid_positions, "Predicted"] = unique_activity_label[unique_codes]
full_predictions.loc[valid_positions, "Probability"] = unique_probability[unique_codes]

prediction_path = ARTIFACT_DIR / "08_case_predictions_all_rows.csv"
full_predictions.to_csv(prediction_path, index=False)
print("LDA training rows:", len(X_reference))
print("Positive training rows:", int((activity_y == 0).sum()))
print("Negative training rows:", int((activity_y == 1).sum()))
print("Saved:", prediction_path)
display(full_predictions.head())

In [ ]:
# STEP 10 — Apply AD-Test thresholds to Case study for k=3...25 and both multipliers.
from sklearn.neighbors import NearestNeighbors

def threshold_token(multiplier):
    return "0p5" if float(multiplier) == 0.5 else "2p0"

summaries = []
detail_paths = {}
result_paths = {}
for multiplier in AD_THRESHOLD_MULTIPLIERS:
    stored_summary = ad_summaries[multiplier].set_index("K")
    detailed = full_predictions.copy()
    professor = pd.DataFrame({
        "Smiles": full_predictions["Original_SMILES"],
        "Predicted": full_predictions["Predicted"],
        "Probability": full_predictions["Probability"],
    })
    for ad_k in AD_NEIGHBOR_K_VALUES:
        knn = NearestNeighbors(n_neighbors=ad_k, metric="euclidean", algorithm="auto")
        knn.fit(X_reference)
        query_distances, _ = knn.kneighbors(X_query_selected)
        unique_mean_distance = query_distances.mean(axis=1)
        threshold = float(stored_summary.loc[f"k={ad_k}", "AD_Distance_Threshold"])
        unique_ind = unique_mean_distance <= threshold

        full_distance = np.full(len(case_frame), np.nan, dtype=float)
        full_label = np.full(len(case_frame), "INVALID", dtype=object)
        full_distance[valid_positions] = unique_mean_distance[unique_codes]
        full_label[valid_positions] = np.where(unique_ind[unique_codes], "IND", "OOD")
        detailed[f"k{ad_k}_Mean_Distance"] = full_distance
        detailed[f"k{ad_k}_Threshold"] = threshold
        detailed[f"ADk{ad_k}"] = full_label
        professor[f"ADk{ad_k}"] = full_label

        ind_count = int(np.sum(full_label == "IND"))
        ood_count = int(np.sum(full_label == "OOD"))
        invalid_count = int(np.sum(full_label == "INVALID"))
        summaries.append({
            "Dataset": RUN_DATASET,
            "FS_Method": RUN_METHOD,
            "Threshold_Multiplier": float(multiplier),
            "k": int(ad_k),
            "Distance_Threshold": threshold,
            "N_Case_Total": int(len(case_frame)),
            "N_Valid": int(valid_mask.sum()),
            "N_Invalid": invalid_count,
            "IND_Count": ind_count,
            "OOD_Count": ood_count,
            "IND_Coverage_Valid": float(ind_count / valid_mask.sum()),
            "IND_Coverage_All": float(ind_count / len(case_frame)),
        })

    token = threshold_token(multiplier)
    detail_path = OUTPUT_DIR / f"case_study_predictions_ood_threshold_{token}.csv"
    result_path = OUTPUT_DIR / f"IND_Result_threshold_{token}.csv"
    detailed.to_csv(detail_path, index=False)
    professor.to_csv(result_path, index=True)
    detail_paths[multiplier] = detail_path
    result_paths[multiplier] = result_path
    print(f"Multiplier={multiplier}: saved {detail_path.name} and {result_path.name}")

summary = pd.DataFrame(summaries)
summary_path = OUTPUT_DIR / "ood_summary_k3_k25.csv"
summary.to_csv(summary_path, index=False)

# Compatibility names use the historical multiplier 0.5.
full_05 = pd.read_csv(detail_paths[0.5])
ind_05 = pd.read_csv(result_paths[0.5], index_col=0)
full_05.to_csv(OUTPUT_DIR / "production_predictions_ood.csv", index=False)
ind_05.to_csv(OUTPUT_DIR / "IND_Result.csv", index=True)

print("Combined summary:", summary_path)
display(summary.head())

In [ ]:
# STEP 11 — Plot coverage and write provenance plus SHA-256 manifest.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
for multiplier, group in summary.groupby("Threshold_Multiplier", sort=True):
    ax.plot(group["k"], group["IND_Coverage_Valid"], marker="o", label=f"multiplier={multiplier}")
ax.set(
    title=f"{RUN_DATASET} {RUN_METHOD}: Case-study IND coverage",
    xlabel="k",
    ylabel="IND coverage among valid Case-study rows",
)
ax.set_xticks(range(3, 26, 2))
ax.set_ylim(0.0, 1.05)
ax.grid(alpha=0.25)
ax.legend()
figure_path = OUTPUT_DIR / "ood_coverage_diagnostics.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

metadata = {
    "dataset": RUN_DATASET,
    "method": RUN_METHOD,
    "case_study_csv": str(INPUT_PATHS["case_study_csv"]),
    "source_ad_output": str(AD_INPUT_DIR),
    "source_ad_metadata_sha256": sha256_file(ad_metadata_path),
    "ad_reference_sha256": sha256_file(ad_reference_path),
    "case_rows": int(len(case_frame)),
    "valid_case_rows": int(valid_mask.sum()),
    "invalid_case_rows": int((~valid_mask).sum()),
    "unique_valid_smiles": int(len(unique_smiles)),
    "selected_feature_count": int(EXPECTED_SELECTED_K),
    "selected_indices": selected_indices.tolist(),
    "threshold_multipliers": list(AD_THRESHOLD_MULTIPLIERS),
    "ad_k_values": list(AD_NEIGHBOR_K_VALUES),
    "observed_case_targets_available": False,
    "regression_metrics_reported": False,
    "activity_threshold_pIC50": ACTIVITY_PIC50_THRESHOLD,
    "invalid_policy": "retain every row; invalid/missing SMILES are marked INVALID",
    "duplicate_policy": "featurize each unique valid SMILES once; map back to every source row",
}
metadata_path = OUTPUT_DIR / "production_manifest.json"
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

artifact_paths = sorted(
    path for path in OUTPUT_DIR.rglob("*")
    if path.is_file() and path.name != "artifact_manifest.csv"
)
manifest = pd.DataFrame([
    {
        "Relative_Path": str(path.relative_to(OUTPUT_DIR)),
        "Bytes": path.stat().st_size,
        "SHA256": sha256_file(path),
    }
    for path in artifact_paths
])
manifest_path = OUTPUT_DIR / "artifact_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print("Coverage plot:", figure_path)
print("Production manifest:", metadata_path)
print("Artifact manifest:", manifest_path)
print("Total output files:", len(manifest) + 1)
display(manifest)

## Interpretation

`Predicted_pIC50` is frozen regression output. `Predicted` and `Probability` come from the Train+Validation-only LDA activity layer. `ADk3`...`ADk25` are independent applicability warnings. `INVALID` means the source SMILES could not be featurized and was deliberately retained for audit.